In [8]:
! pip install python-igraph

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
    --------------------------------------- 0.0/2.0 MB 1.4 MB/s eta 0:00:02
    --------------------------------------- 0.0/2.0 MB 1.4 MB/s eta 0:00:02
    --------------------------------------- 0.0/2.0 MB 281.8 kB/s eta 0:00:07
   -- ------------------------------------- 0.1/2.0 MB 656.4 kB/s eta 0:00:03
   -- ------------------------------------- 0.1/2.0 MB 656.4 kB/s eta 0:00:03
   --- ------------------------------------ 0.2/2.0 MB 695.5 kB/s eta 0:00:03
   ----- ---------------------------------- 0.3/2.0 MB 899.5 kB/s eta 0:00:02
   ------ --------------------------------- 0.3/2.0 MB 925.5 kB/s eta 0:00:02
   --------- ------------------------------ 0.5/2.0 MB 1.2 MB/s eta 0:00:02
   ----------- ---------------------------- 0.6/2.0 MB 1.3 MB/s eta 0:00:02
   --------------- ------------------------ 0.8/2.0 MB 1.6 MB/s eta 0:00:01
   ---------------------- ----------------- 1.1/2.0 MB 2.1 MB/s eta 0:00:01
   ----


[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import re
import nltk
import gensim
import pandas as pd
import networkx as nx
from gensim import corpora
from pymongo import MongoClient
from nltk.corpus import stopwords
import plotly.graph_objects as go
from pyvis.network import Network
from collections import defaultdict
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [2]:
client = MongoClient("mongodb://localhost:27017/")
db = client['DataTails']
collection = db['Data']  
data_cursor = collection.find({})
DF = pd.DataFrame(list(data_cursor))
print(DF.head())

                        _id  type  \
0  66e965e698330736e0d693d5  None   
1  66e965e698330736e0d693d6  None   
2  66e965e698330736e0d693d7  None   
3  66e965e698330736e0d693d8  None   
4  66e965e698330736e0d693d9  None   

                                           postTitle postDesc  \
0  Adults(especially those over 30), how young do...      NaN   
1  What is a thing that your parents consider nor...      NaN   
2                 What is a smell that comforts you?      NaN   
3  When in history was it the best time to be a w...      NaN   
4    What's the worst way someone broke up with you?      NaN   

              postTime            authorName noOfUpvotes isNSFW  \
0  2024-08-06 01:02:35  Excellent-Studio7257        4068  False   
1  2024-08-06 01:47:22        Bigbumoffhappy        2073  False   
2  2024-08-05 22:21:53         bloomsmittenn        2188  False   
3  2024-08-06 03:32:59   More_food_please_77         778  False   
4  2024-08-05 21:01:39    ImpressiveWrap7363       

In [3]:
lemmatizer = WordNetLemmatizer()
StopWords = set(stopwords.words('english'))
def Preprocessing(text):
    text = re.sub(r'\W', ' ', text)
    tokens = word_tokenize(text.lower())
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in StopWords]
    return ' '.join(tokens)

Cols = ['subReddit', 'postTitle', 'postDesc', 'postTime', 'authorName', 'noOfUpvotes', 'comments', 'noOfComments', 'postUrl']
DF = DF[Cols]
print(DF.isnull().sum())
DF['postDesc'].fillna('', inplace=True)
DF['noOfUpvotes'].fillna(0, inplace=True)
DF['postTime'] = pd.to_datetime(DF['postTime'], errors='coerce')
DF['comments'] = DF['comments'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))
DF['postTitle'] = DF['postTitle'].apply(lambda x: Preprocessing(str(x)))
DF['postDesc'] = DF['postDesc'].apply(lambda x: Preprocessing(str(x)))
print(DF.head())
print(DF.isnull().sum())

subReddit           2
postTitle           0
postDesc        15607
postTime            0
authorName        587
noOfUpvotes         0
comments            0
noOfComments        2
postUrl             2
dtype: int64
   subReddit                                          postTitle postDesc  \
0  AskReddit                  adult especially 30 young 18 seem            
1  AskReddit  thing parent consider normal consider normal a...            
2  AskReddit                                      smell comfort            
3  AskReddit                            history best time woman            
4  AskReddit                            worst way someone broke            

             postTime            authorName noOfUpvotes  \
0 2024-08-06 01:02:35  Excellent-Studio7257        4068   
1 2024-08-06 01:47:22        Bigbumoffhappy        2073   
2 2024-08-05 22:21:53         bloomsmittenn        2188   
3 2024-08-06 03:32:59   More_food_please_77         778   
4 2024-08-05 21:01:39    ImpressiveWr

In [4]:
GroupedData = DF.groupby('subReddit')
SubReddits = defaultdict(dict)
for subreddit, group in GroupedData:
    print(f"Processing Subreddit: {subreddit}")
    group['Combined'] = group['postTitle'] + " " + group['postDesc']
    group['Tokens'] = group['Combined'].apply(lambda x: word_tokenize(x))
    dictionary = corpora.Dictionary(group['Tokens'])
    corpus = [dictionary.doc2bow(text) for text in group['Tokens']]
    LDA = gensim.models.ldamodel.LdaModel(corpus, num_topics=5, id2word=dictionary, passes=15)
    SubReddits[subreddit]['lda_model'] = LDA
    SubReddits[subreddit]['dictionary'] = dictionary
    SubReddits[subreddit]['corpus'] = corpus
    group['topics'] = [LDA.get_document_topics(bow) for bow in corpus]
    DF.loc[group.index, 'topics'] = group['topics']

Processing Subreddit: Advice
Processing Subreddit: AmItheAsshole
Processing Subreddit: AskReddit
Processing Subreddit: Damnthatsinteresting
Processing Subreddit: Filmmakers
Processing Subreddit: Jokes
Processing Subreddit: Music
Processing Subreddit: NoStupidQuestions
Processing Subreddit: Showerthoughts
Processing Subreddit: askscience
Processing Subreddit: aww
Processing Subreddit: books
Processing Subreddit: funny
Processing Subreddit: gadgets
Processing Subreddit: gaming
Processing Subreddit: help
Processing Subreddit: islamabad
Processing Subreddit: memes
Processing Subreddit: mildlyinteresting
Processing Subreddit: movies
Processing Subreddit: news
Processing Subreddit: olympics
Processing Subreddit: pakistan
Processing Subreddit: pics
Processing Subreddit: politics
Processing Subreddit: programming
Processing Subreddit: science
Processing Subreddit: showerthoughts
Processing Subreddit: socialmedia
Processing Subreddit: sports
Processing Subreddit: technology
Processing Subreddit

In [5]:
print(DF.shape)
print(DF.describe())
print(DF.head())


(24072, 10)
       noOfComments
count  24070.000000
mean     578.256543
std     1389.473933
min        0.000000
25%        5.000000
50%       27.000000
75%      618.000000
max    38118.000000
   subReddit                                          postTitle postDesc  \
0  AskReddit                  adult especially 30 young 18 seem            
1  AskReddit  thing parent consider normal consider normal a...            
2  AskReddit                                      smell comfort            
3  AskReddit                            history best time woman            
4  AskReddit                            worst way someone broke            

             postTime            authorName noOfUpvotes  \
0 2024-08-06 01:02:35  Excellent-Studio7257        4068   
1 2024-08-06 01:47:22        Bigbumoffhappy        2073   
2 2024-08-05 22:21:53         bloomsmittenn        2188   
3 2024-08-06 03:32:59   More_food_please_77         778   
4 2024-08-05 21:01:39    ImpressiveWrap7363        1989 

## D3 Code

### Version one

In [ ]:
# <!DOCTYPE html>
# <html lang="en">
# <head>
#     <meta charset="UTF-8">
#     <meta name="viewport" content="width=device-width, initial-scale=1.0">
#     <title>Reddit Knowledge Graph</title>
#     <style>
#         .node { stroke: #fff; stroke-width: 1.5px; }
#         .link { stroke: #999; stroke-opacity: 0.6; }
#         text { font-family: sans-serif; font-size: 12px; }
#     </style>
# </head>
# <body>
#     <script src="https://d3js.org/d3.v6.min.js"></script>
#     <script>
#         // Load the graph data from the generated JSON file
#         d3.json("reddit_graph_with_ontologies.json").then(function(graph) {
#             const width = 960, height = 600;

#             const svg = d3.select("body").append("svg")
#                 .attr("width", width)
#                 .attr("height", height);

#             // Set up the force simulation
#             const simulation = d3.forceSimulation(graph.nodes)
#                 .force("link", d3.forceLink(graph.links).id(d => d.id).distance(100))
#                 .force("charge", d3.forceManyBody().strength(-300))
#                 .force("center", d3.forceCenter(width / 2, height / 2));

#             // Draw links
#             const link = svg.append("g")
#                 .selectAll("line")
#                 .data(graph.links)
#                 .enter().append("line")
#                 .attr("class", "link");

#             // Draw nodes
#             const node = svg.append("g")
#                 .selectAll("circle")
#                 .data(graph.nodes)
#                 .enter().append("circle")
#                 .attr("r", 10)
#                 .attr("class", "node")
#                 .style("fill", d => {
#                     if (d.type === "sioc:Post") return "blue";
#                     if (d.type === "sioc:Container") return "green";
#                     if (d.type === "foaf:Person") return "red";
#                     if (d.type === "sioc:Comment") return "purple";
#                     return "yellow";
#                 })
#                 .call(d3.drag()
#                     .on("start", dragstarted)
#                     .on("drag", dragged)
#                     .on("end", dragended));

#             // Add labels to nodes
#             node.append("title").text(d => `${d.id} (${d.type})`);

#             // Update positions on simulation tick
#             simulation.on("tick", () => {
#                 link.attr("x1", d => d.source.x)
#                     .attr("y1", d => d.source.y)
#                     .attr("x2", d => d.target.x)
#                     .attr("y2", d => d.target.y);

#                 node.attr("cx", d => d.x)
#                     .attr("cy", d => d.y);
#             });

#             // Drag functions
#             function dragstarted(event, d) {
#                 if (!event.active) simulation.alphaTarget(0.3).restart();
#                 d.fx = d.x;
#                 d.fy = d.y;
#             }

#             function dragged(event, d) {
#                 d.fx = event.x;
#                 d.fy = event.y;
#             }

#             function dragended(event, d) {
#                 if (!event.active) simulation.alphaTarget(0);
#                 d.fx = null;
#                 d.fy = null;
#             }
#         });
#     </script>
# </body>
# </html>


### Version 2

In [ ]:
# <!DOCTYPE html>
# <html lang="en">
# <head>
#     <meta charset="UTF-8">
#     <title>Reddit Knowledge Graph</title>
#     <script src="https://d3js.org/d3.v5.min.js"></script>
#     <style>
#         .node circle {
#             fill: #69b3a2;
#             stroke: #333;
#             stroke-width: 1.5px;
#         }
#         .link {
#             stroke: #999;
#             stroke-opacity: 0.6;
#         }
#     </style>
# </head>
# <body>

# <svg width="960" height="600"></svg>

# <script>
#     // Load the JSON file
#     d3.json("reddit_knowledge_graph.json").then(function(graph) {

#         var svg = d3.select("svg"),
#             width = +svg.attr("width"),
#             height = +svg.attr("height");

#         var simulation = d3.forceSimulation()
#             .force("link", d3.forceLink().id(function(d) { return d.id; }))
#             .force("charge", d3.forceManyBody().strength(-200))
#             .force("center", d3.forceCenter(width / 2, height / 2));

#         // Add links (edges)
#         var link = svg.append("g")
#             .attr("class", "links")
#             .selectAll("line")
#             .data(graph.edges)
#             .enter().append("line")
#             .attr("class", "link");

#         // Add nodes
#         var node = svg.append("g")
#             .attr("class", "nodes")
#             .selectAll("g")
#             .data(graph.nodes)
#             .enter().append("g");

#         var circles = node.append("circle")
#             .attr("r", 10)
#             .attr("fill", function(d) { return d.group === "Post" ? "blue" : d.group === "Subreddit" ? "green" : "red"; });

#         var labels = node.append("text")
#             .text(function(d) { return d.label; })
#             .attr('x', 15)
#             .attr('y', 3);

#         simulation
#             .nodes(graph.nodes)
#             .on("tick", ticked);

#         simulation.force("link")
#             .links(graph.edges);

#         function ticked() {
#             link
#                 .attr("x1", function(d) { return d.source.x; })
#                 .attr("y1", function(d) { return d.source.y; })
#                 .attr("x2", function(d) { return d.target.x; })
#                 .attr("y2", function(d) { return d.target.y; });

#             node
#                 .attr("transform", function(d) {
#                     return "translate(" + d.x + "," + d.y + ")";
#                 });
#         }

#     });
# </script>
# </body>
# </html>
